In [1]:
%%capture
!pip install ortools

In [2]:
from ortools.linear_solver import pywraplp

# Dataset: [("Operation", SMV, Quantity)]
operations = [
    ("Collar Stitching", 0.5, 500),
    ("Sleeve Attachment", 0.7, 500),
    ("Side Seam", 0.6, 500),
    ("Hemming", 0.4, 500),
    ("Button & Finishing", 0.9, 500),
]

SHIFT_TIME = 480  # minutes per operator per shift

# Create solver (Integer Programming)
solver = pywraplp.Solver.CreateSolver('SCIP')

x = {}  # operator variables
for i, (name, smv, qty) in enumerate(operations):
    x[i] = solver.IntVar(0, solver.infinity(), f"x_{i}")

# Constraints: Each operation must have enough capacity
for i, (name, smv, qty) in enumerate(operations):
    workload = smv * qty
    solver.Add(x[i] * SHIFT_TIME >= workload)

# Objective: Minimize total operators
solver.Minimize(solver.Sum([x[i] for i in x]))

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Optimal solution found!\n")
    total_ops = 0
    for i, (name, smv, qty) in enumerate(operations):
        print(f"{name}: {int(x[i].solution_value())} operators")
        total_ops += int(x[i].solution_value())
    print(f"\nTotal operators required: {total_ops}")
else:
    print("No optimal solution found.")

Optimal solution found!

Collar Stitching: 1 operators
Sleeve Attachment: 1 operators
Side Seam: 1 operators
Hemming: 1 operators
Button & Finishing: 1 operators

Total operators required: 5


In [3]:
from ortools.sat.python import cp_model

# Dataset
operations = [
    ("Collar Stitching", 0.5, 500, "Lockstitch"),
    ("Sleeve Attachment", 0.7, 500, "Overlock"),
    ("Side Seam", 0.6, 500, "Overlock"),
    ("Hemming", 0.4, 500, "Flatlock"),
    ("Button & Finishing", 0.9, 500, "Buttonhole"),
]

SHIFT_TIME = 480

operators = {
    "Op1": ["Lockstitch", "Overlock"],
    "Op2": ["Overlock", "Flatlock"],
    "Op3": ["Buttonhole", "Lockstitch"],
    "Op4": ["Lockstitch", "Overlock", "Flatlock"],
    "Op5": ["Buttonhole"],
}

# Calculate workloads
workloads = [smv * qty for _, smv, qty, _ in operations]

# Create CP-SAT model
model = cp_model.CpModel()

# Decision variables: assign[op_id][operator_id] = workload minutes assigned
assign = {}
for i, (op_name, smv, qty, machine) in enumerate(operations):
    for j, op_name2 in enumerate(operators.keys()):
        assign[(i, j)] = model.NewIntVar(0, SHIFT_TIME, f"assign_{i}_{j}")

# Constraints: each operation's workload must be met by skilled operators
for i, (op_name, smv, qty, machine) in enumerate(operations):
    skilled_ops = [j for j, skills in enumerate(operators.values()) if machine in skills]
    model.Add(sum(assign[(i, j)] for j in skilled_ops) >= workloads[i])

# Constraints: each operator can’t exceed shift time
for j, skills in enumerate(operators.values()):
    model.Add(sum(assign[(i, j)] for i in range(len(operations))) <= SHIFT_TIME)

# Objective: minimize total idle time (maximize utilization)
total_idle = []
for j in range(len(operators)):
    total_idle.append(SHIFT_TIME - sum(assign[(i, j)] for i in range(len(operations))))
model.Minimize(sum(total_idle))

# Solve
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 10
status = solver.Solve(model)

if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
    print("Solution found!\n")
    for i, (op_name, smv, qty, machine) in enumerate(operations):
        print(f"{op_name} ({machine}, {workloads[i]} mins):")
        for j, op_name2 in enumerate(operators.keys()):
            val = solver.Value(assign[(i, j)])
            if val > 0:
                print(f"  - {op_name2} assigned {val} mins")
    print("\nOperator utilizations:")
    for j, op_name2 in enumerate(operators.keys()):
        used = sum(solver.Value(assign[(i, j)]) for i in range(len(operations)))
        print(f"{op_name2}: {used}/{SHIFT_TIME} mins used ({used/SHIFT_TIME*100:.1f}%)")
else:
    print("No feasible solution found.")

TypeError: __ge__(): incompatible function arguments. The following argument types are supported:
    1. (self: ortools.sat.python.cp_model_helper.LinearExpr, arg0: ortools.sat.python.cp_model_helper.LinearExpr) -> operations_research::sat::python::BoundedLinearExpression
    2. (self: ortools.sat.python.cp_model_helper.LinearExpr, arg0: int) -> operations_research::sat::python::BoundedLinearExpression

Invoked with: SumArray(assign_0_0(0..480), assign_0_2(0..480), assign_0_3(0..480)), 250.0